# Defi du jour — Agent minuscule avec outils (smolagents)Objectif : creer un agent debutant capable d'appeler des outils simples avec `smolagents`.**Aucune API payante, aucune clef.** Modele local minimal par defaut.## Comment utiliser ce notebook- Les cellules **EXERCICE** contiennent des `TODO` : c'est a vous.- Sous chacune, une cellule **Solution** est repliee. Double-cliquez sur son titre pour l'afficher.- Executez soit votre version, soit la solution — la derniere executee ecrase la precedente.## Regles de reponse imposees- 2 a 4 phrases maximum.- Balise source `[kb:N]` des que la base de connaissances est utilisee.- Sans preuve : le dire explicitement et proposer une question de suivi.## Ce que vous allez construire1. Une base de connaissances (`kb_snippets`) de 5 a 8 extraits courts.2. `KBLookupTool` — renvoie les extraits contenant les mots-cles.3. `MathTool` — additionne ou multiplie deux nombres (`a`, `b`, `op`).4. Un modele local (`TransformersModel` / `sshleifer/tiny-gpt2`) ou un stub.5. Un `ToolCallingAgent`, teste sur 3 requetes.

## 1. Installation

In [ ]:
!pip install -q "smolagents[transformers]" wikipedia

In [ ]:
import warningswarnings.filterwarnings("ignore")from smolagents import Tool, ToolCallingAgentprint("Imports OK")

## 2. Base de connaissances (`kb_snippets`)**EXERCICE :** definissez 5 a 8 chaines courtes, chacune avec un `id`, une `source` et un `text`.

In [ ]:
# TODO : completez 5 a 8 extraits courts, chacun avec un id, une source et un texte.kb_snippets = [    {"id": 1, "source": "kb:1", "text":     "Une boucle agentique (agentic loop) est le cycle Percevoir -> Planifier -> Agir -> "     "Observer, repete jusqu'a pouvoir repondre."},    {"id": 2, "source": "kb:2", "text":     "Un outil (tool) est une fonction decrite en langage naturel que le modele peut appeler."},    # TODO : ajoutez au moins 3 extraits de plus (smolagents, RAG, hallucination, ReAct...)]print(f"{len(kb_snippets)} extraits charges")

In [ ]:
# @title Solution — kb_snippets { display-mode: "form" }kb_snippets = [    {"id": 1, "source": "kb:1", "text":     "Une boucle agentique (agentic loop) est le cycle Percevoir -> Planifier -> Agir -> "     "Observer. L'agent lit la demande, choisit un outil, execute l'action, observe le "     "resultat, puis recommence jusqu'a pouvoir repondre."},    {"id": 2, "source": "kb:2", "text":     "Un outil (tool) est une fonction decrite en langage naturel que le modele peut appeler. "     "Sa description et ses types d'entree guident le LLM pour choisir le bon outil."},    {"id": 3, "source": "kb:3", "text":     "smolagents est un framework minimaliste de Hugging Face pour construire des agents. "     "Il fournit ToolCallingAgent (appels d'outils en JSON) et CodeAgent (l'agent ecrit du code Python)."},    {"id": 4, "source": "kb:4", "text":     "Le RAG (Retrieval-Augmented Generation) recupere des passages pertinents avant de generer "     "la reponse, ce qui ancre le texte dans des sources et reduit les hallucinations."},    {"id": 5, "source": "kb:5", "text":     "Une hallucination est une affirmation fluide mais fausse produite par un LLM. Citer ses "     "sources et ancrer la generation dans des documents recuperes est la principale parade."},    {"id": 6, "source": "kb:6", "text":     "TransformersModel permet a smolagents d'utiliser un modele local Hugging Face, sans clef "     "API. Exemple : sshleifer/tiny-gpt2, minuscule et rapide mais de tres faible qualite."},    {"id": 7, "source": "kb:7", "text":     "Le ReAct (Reasoning + Acting) alterne raisonnement et actions : le modele ecrit une pensee, "     "appelle un outil, lit l'observation, puis conclut."},    {"id": 8, "source": "kb:8", "text":     "Un agent doit s'arreter : on limite le nombre de tours (max_steps) pour eviter les boucles "     "infinies quand aucun outil ne resout la tache."},]print(f"{len(kb_snippets)} extraits charges")

## 3. `KBLookupTool`**EXERCICE :** sous-classez `Tool` pour renvoyer les extraits correspondants.La `description` est lue par le LLM : soyez precis sur *quand* utiliser l'outil.

In [ ]:
class KBLookupTool(Tool):    name = "kb_lookup"    description = (        # TODO : decrivez PRECISEMENT quand utiliser cet outil.        # La description guide le LLM dans son choix d'outil.        "..."    )    inputs = {        "query": {"type": "string", "description": "TODO : decrire l'argument."}    }    output_type = "string"    def forward(self, query: str) -> str:        # TODO :        # 1. decouper `query` en mots-cles (minuscules, ignorer les mots vides et les mots <= 2 lettres)        # 2. compter, pour chaque extrait, le nombre de mots-cles presents dans son texte        # 3. trier par score decroissant, garder les 3 meilleurs        # 4. renvoyer une chaine "[kb:N] texte" par ligne        # 5. si aucun resultat : renvoyer un message AUCUNE_PREUVE        raise NotImplementedErrorkb_tool = KBLookupTool()print(kb_tool.forward("boucle agentique"))

In [ ]:
# @title Solution — KBLookupTool { display-mode: "form" }class KBLookupTool(Tool):    name = "kb_lookup"    description = (        "Cherche dans la base de connaissances interne sur l'IA agentique, les outils, "        "smolagents, le RAG et les hallucinations. Renvoie les extraits correspondants "        "avec leur balise source [kb:N]."    )    inputs = {        "query": {            "type": "string",            "description": "Mots-cles a rechercher, par exemple 'boucle agentique'.",        }    }    output_type = "string"    def forward(self, query: str) -> str:        STOP = {"une", "un", "le", "la", "les", "des", "de", "du", "est", "que",                "qu", "quoi", "ce", "quest", "en", "et", "the", "what", "is", "a"}        words = [w for w in "".join(c if c.isalnum() else " " for c in query.lower()).split()                 if len(w) > 2 and w not in STOP]        hits = []        for s in kb_snippets:            low = s["text"].lower()            score = sum(1 for w in words if w in low)            if score:                hits.append((score, s))        hits.sort(key=lambda x: -x[0])        hits = hits[:3]        if not hits:            return ("AUCUNE_PREUVE : rien dans la base de connaissances ne correspond a "                    f"'{query}'. Dis-le a l'utilisateur et propose une question de suivi.")        return "\n".join(f"[{s['source']}] {s['text']}" for _, s in hits)kb_tool = KBLookupTool()print(kb_tool.forward("boucle agentique"))

## 4. `MathTool`**EXERCICE :** addition / multiplication via les arguments `a`, `b`, `op`.

In [ ]:
class MathTool(Tool):    name = "math"    description = "TODO : decrire l'outil et les valeurs acceptees pour op."    inputs = {        "a": {"type": "number", "description": "TODO"},        "b": {"type": "number", "description": "TODO"},        "op": {"type": "string", "description": "TODO : 'add' ou 'multiply'"},    }    output_type = "string"    def forward(self, a: float, b: float, op: str) -> str:        # TODO :        # - si op vaut 'add' -> renvoyer "a + b = resultat"        # - si op vaut 'multiply' -> renvoyer "a * b = resultat"        # - sinon -> renvoyer un message d'erreur clair        raise NotImplementedErrormath_tool = MathTool()print(math_tool.forward(12, 30, "add"))      # attendu : 12 + 30 = 42print(math_tool.forward(7, 6, "multiply"))   # attendu : 7 * 6 = 42

In [ ]:
# @title Solution — MathTool { display-mode: "form" }class MathTool(Tool):    name = "math"    description = (        "Effectue une operation arithmetique sur deux nombres. "        "Utilise op='add' pour additionner, op='multiply' pour multiplier."    )    inputs = {        "a": {"type": "number", "description": "Premier nombre."},        "b": {"type": "number", "description": "Deuxieme nombre."},        "op": {"type": "string", "description": "Operation : 'add' ou 'multiply'."},    }    output_type = "string"    def forward(self, a: float, b: float, op: str) -> str:        a, b = float(a), float(b)        op = str(op).strip().lower()        if op in ("add", "+", "addition", "sum"):            r, sym = a + b, "+"        elif op in ("multiply", "*", "mul", "x", "times"):            r, sym = a * b, "*"        else:            return f"Erreur : operation '{op}' inconnue. Utilise 'add' ou 'multiply'."        if r == int(r):            r = int(r)        return f"{a:g} {sym} {b:g} = {r}"math_tool = MathTool()print(math_tool.forward(12, 30, "add"))print(math_tool.forward(7, 6, "multiply"))

## 5. Modele : `TransformersModel` ou stub**EXERCICE :** branchez un modele.`sshleifer/tiny-gpt2` (~2M parametres) est le modele demande par la consigne, mais il esttrop petit pour emettre le JSON d'appel d'outil attendu : l'agent echouera. C'est instructifa observer une fois (`USE_TINY_GPT2 = True`).Pour que la boucle agentique soit reellement visible, la solution fournit un **stubdeterministe sans etat** : il relit `messages` a chaque tour et applique des regles simples.Pour un vrai comportement local : `Qwen/Qwen2.5-1.5B-Instruct`.

In [ ]:
# TODO : choisissez votre modele.## Option A (recommandee) — stub deterministe : recopiez la classe StubModel du notebook#   professeur, ou ecrivez la votre. Elle doit renvoyer un ChatMessage avec tool_calls.## Option B — vrai modele local :#   from smolagents import TransformersModel#   model = TransformersModel(model_id="sshleifer/tiny-gpt2", max_new_tokens=128)#   Attention : tiny-gpt2 (~2M parametres) ne produira pas d'appels d'outils valides.#   Observez l'echec, c'est le but pedagogique.model = None  # TODOprint("Modele :", model)

In [ ]:
# @title Solution — StubModel / TransformersModel { display-mode: "form" }USE_TINY_GPT2 = False  # True = vrai modele local (tres faible), False = stub deterministeif USE_TINY_GPT2:    from smolagents import TransformersModel    model = TransformersModel(model_id="sshleifer/tiny-gpt2", max_new_tokens=128)    print("Modele local : sshleifer/tiny-gpt2")else:    import re    from smolagents.models import Model, ChatMessage    class StubModel(Model):        """Stub deterministe : joue le role du LLM en emettant de vrais appels d'outils.        tiny-gpt2 (~2M parametres) est incapable de produire le JSON d'appel d'outil        attendu. Ce stub applique des regles simples pour rendre la boucle agentique        Percevoir -> Agir -> Observer reellement observable.        Il est SANS ETAT : il relit l'historique `messages` a chaque tour, donc il        fonctionne sur plusieurs `agent.run()` successifs.        """        def __init__(self):            super().__init__(model_id="stub")        # --- lecture de l'historique -------------------------------------        @staticmethod        def _texts(messages):            """Renvoie [(role, texte), ...] a plat."""            out = []            for m in messages:                role = getattr(m.role, "value", str(m.role))                content = m.content                if isinstance(content, list):                    txt = " ".join(str(c.get("text", "")) for c in content)                else:                    txt = str(content or "")                out.append((role, txt))            return out        def _task(self, messages):            """La tache = le PREMIER message utilisateur (pas le system prompt)."""            for role, txt in self._texts(messages):                if role == "user":                    return txt.replace("New task:", " ").strip()                                return ""        def _observations(self, messages):            """Concatene les retours d'outils deja recus (messages 'tool-response')."""            return "\n".join(txt for role, txt in self._texts(messages)                              if "tool" in role.lower())        # --- boucle de decision ------------------------------------------        def generate(self, messages, stop_sequences=None, tools_to_call_from=None, **kw):            task = self._task(messages)            obs = self._observations(messages)            # Tour 2+ : un outil a deja repondu -> rediger la reponse finale.            if obs.strip():                return self._call("final_answer", {"answer": self._compose(obs)})            # Tour 1 : choisir l'outil a partir de la tache uniquement.            low = task.lower()            nums = [float(x) for x in re.findall(r"-?\d+(?:[.,]\d+)?", low.replace(",", "."))]            if len(nums) >= 2 and any(k in low for k in                                      ("multipli", "multiply", "fois", "times", "*")):                return self._call("math", {"a": nums[0], "b": nums[1], "op": "multiply"})            if len(nums) >= 2 and any(k in low for k in                                      ("ajout", "add", "additionn", "somme", "plus", "+")):                return self._call("math", {"a": nums[0], "b": nums[1], "op": "add"})            return self._call("kb_lookup", {"query": task})        @staticmethod        def _call(name, args):            return ChatMessage(role="assistant", content=None, tool_calls=[{                "id": "call_1", "type": "function",                "function": {"name": name, "arguments": args},            }])        # --- redaction de la reponse finale (2 a 4 phrases + source) ------        @staticmethod        def _compose(obs):            if "AUCUNE_PREUVE" in obs:                return ("Je n'ai trouve aucun element pertinent dans la base de connaissances "                        "pour repondre. Question de suivi : pouvez-vous preciser le terme exact "                        "ou le domaine concerne ?")            # Cas base de connaissances -> citer [kb:N]            m = re.search(r"\[(kb:\d+)\]\s*(.+)", obs, re.S)            if m:                tag, body = m.group(1), m.group(2).strip()                body = re.split(r"\n\[kb:", body)[0].strip()                phrases = re.split(r"(?<=\.)\s+", body)[:3]                return " ".join(phrases).strip() + f" [{tag}]"            # Cas calcul -> reprendre le resultat de l'outil math            m = re.search(r"(-?[\d.]+\s*[+*]\s*-?[\d.]+\s*=\s*-?[\d.]+)", obs)            if m:                return (f"Le resultat est {m.group(1)}. "                        "Le calcul a ete effectue par l'outil math.")            return ("Je n'ai pas assez d'elements pour repondre avec certitude. "                    "Pouvez-vous reformuler votre question ?")    model = StubModel()    print("Modele : StubModel (deterministe, sans etat)")

## 6. `ToolCallingAgent`**EXERCICE :** instanciez l'agent avec vos deux outils, votre modele et `max_steps=3`.

In [ ]:
# TODO : instanciez le ToolCallingAgent avec vos deux outils, votre modele et max_steps=3.agent = None  # TODO# print("Outils disponibles :", [t for t in agent.tools])

In [ ]:
# @title Solution — ToolCallingAgent { display-mode: "form" }agent = ToolCallingAgent(    tools=[kb_tool, math_tool],    model=model,    max_steps=3,)print("Outils disponibles :", [t for t in agent.tools])

## 7. Trois requetes de test**EXERCICE :** imprimez le plan d'appels d'outils **et** la reponse finale de chaque requete.Verifiez que l'agent cite `[kb:N]` quand il utilise la base de connaissances.

In [ ]:
QUERIES = [    "Ajoutez 12 et 30.",    "Multipliez 7 par 6.",    "Qu'est-ce qu'une boucle d'IA agentique ?",]for i, q in enumerate(QUERIES, 1):    print("=" * 76)    print(f"REQUETE {i} : {q}")    print("=" * 76)    # TODO : appelez agent.run(q)    # TODO : parcourez agent.memory.steps et imprimez chaque tool_call (nom + arguments)    #        ainsi que l'observation renvoyee    # TODO : imprimez la reponse finale    pass

In [ ]:
# @title Solution — execution des 3 requetes { display-mode: "form" }QUERIES = [    "Ajoutez 12 et 30.",    "Multipliez 7 par 6.",    "Qu'est-ce qu'une boucle d'IA agentique ?",]for i, q in enumerate(QUERIES, 1):    print("=" * 76)    print(f"REQUETE {i} : {q}")    print("=" * 76)    result = agent.run(q)    print("\n--- APPELS D'OUTILS ---")    for step in agent.memory.steps:        for tc in (getattr(step, "tool_calls", None) or []):            print(f"  -> {tc.name}({tc.arguments})")        obs = getattr(step, "observations", None)        if obs:            print(f"     observation : {str(obs)[:160].strip()}")    print("\n--- REPONSE FINALE ---")    print(f"  {result}\n")

## 8. Bonus — absence de preuveL'agent doit reconnaitre qu'il ne sait pas, plutot que d'inventer.

In [ ]:
print(agent.run("Quelle est la capitale de la Mongolie ?"))

## Verification- [ ] `kb_snippets` contient 5 a 8 extraits avec une source- [ ] `KBLookupTool` renvoie les extraits avec la balise `[kb:N]`- [ ] `MathTool` gere `add` et `multiply`- [ ] L'agent tourne sur les 3 requetes- [ ] Les appels d'outils sont visibles (nom + arguments + observation)- [ ] La reponse issue de la KB cite `[kb:N]`- [ ] Sans preuve, l'agent le dit et propose une question de suivi## Resultats attendus| Requete | Appel d'outil | Reponse finale ||---|---|---|| Ajoutez 12 et 30. | `math(a=12, b=30, op='add')` | Le resultat est 12 + 30 = 42 || Multipliez 7 par 6. | `math(a=7, b=6, op='multiply')` | Le resultat est 7 * 6 = 42 || Qu'est-ce qu'une boucle d'IA agentique ? | `kb_lookup(query=...)` | Percevoir -> Planifier -> Agir -> Observer... **[kb:1]** |## Pour aller plus loin- Ajouter un outil Wikipedia (`wikipedia`, deja installe).- Passer au `CodeAgent` : l'agent ecrit du Python au lieu d'appeler du JSON.- Remplacer le stub par `Qwen/Qwen2.5-1.5B-Instruct` via `TransformersModel`.